# Module 5: Optimize the Server Through Kubernetes

In Module 4 you drove rising load until the batch filled, the queue backed up, and the KV cache ran out, and you named the bottleneck in the metrics. This is the module that pays it off. When the server is yours, that bottleneck is not a wall a provider set, it is a config line you can change. In this module you edit the [vLLM](https://docs.vllm.ai) manifest, redeploy through Kubernetes, and re-run the exact same load sweep to prove you moved the knee. No model change, no new hardware, just better use of the GPU you already have. This is the heart of the workshop.

## Learning objectives
- Run the measure, read, change, redeploy, re-measure loop on a server you control
- Read peak throughput, TTFT, KV cache usage, and preemptions to pick the next flag
- Edit the four engine flags in the vLLM manifest one at a time
- Redeploy with `kubectl apply` and `kubectl rollout restart`, then wait for ready
- Raise `--gpu-memory-utilization` and the batch caps and prove the throughput gain
- Decide when you have reached the ceiling for this model and this GPU

## Prerequisites
- Finished Module 4, with a before sweep that shows a knee and a limiting signal
- A live vLLM endpoint and a reachable namespace, both verified in Module 0
- The baseline manifest at `manifests/vllm-baseline.yaml`, deliberately under-tuned
- About 30 minutes. This is the heart of the workshop, so give it the time

References: [vLLM optimization and tuning](https://docs.vllm.ai/en/stable/configuration/optimization/) &middot; [vLLM engine arguments](https://docs.vllm.ai/en/stable/serving/engine_args.html) &middot; [kubectl rollout](https://kubernetes.io/docs/reference/generated/kubectl/kubectl-commands#rollout) &middot; [Kubernetes Deployment strategy](https://kubernetes.io/docs/concepts/workloads/controllers/deployment/#strategy) &middot; [Linode Kubernetes Engine](https://www.linode.com/products/kubernetes/)

## The tuning loop design basics

Tuning an inference server is not a one-shot edit. It is a loop you go around two or three times, and each pass changes exactly one thing so the metrics stay readable.

- **Measure.** Run the load sweep. Record peak throughput, TTFT, peak KV cache, and preemptions.
- **Read.** Let the metrics pick the next move. Cache near 100 percent with preemptions means raise memory. Cache with headroom but flat throughput means raise the batch.
- **Change.** Edit one or two flags in `manifests/vllm-baseline.yaml`. These are engine startup arguments, so a file edit alone changes nothing live.
- **Redeploy.** `kubectl apply` updates the spec, then `kubectl rollout restart` gives you a fresh pod that reads the new arguments. Wait for ready.
- **Re-measure.** Run the same sweep and compare against your before picture.

Change four flags at once and you will not know which one helped. Change one, redeploy, sweep, repeat.

![A tuning loop on a server you own: measure with a sweep, read the metrics, change one engine flag in the manifest, redeploy with kubectl apply and rollout restart, then re-measure, with the four tunable flags listed in a side panel](images/05_optimize_the_server_architecture.png)

## 1. Setup

Install the two packages this module needs to drive load and plot the result. `kubectl` is already on your PATH in the workshop environment. We reinstall here so this notebook stands on its own.

In [ ]:
%pip install -q "openai>=1.40" "matplotlib>=3.7"


## 2. Configure endpoint, namespace, and manifest

`get_settings()` reads `VLLM_HOST`, `MODEL_NAME`, and `NAMESPACE` from your environment (Module 0 set them). You edit the manifest in place and apply it to your namespace. The load helper mirrors Module 4: it uses the `vllm` CLI when it is present, and falls back to the pure-Python load generator in `common/load.py` when it is not, so the same sweep runs in both prerequisite paths.

In [ ]:
# Setup: settings, server root, the metrics URL, and the manifest path.
import os, sys, json, time, shutil, subprocess, threading, tempfile
sys.path.insert(0, os.path.abspath(".."))

from common.config import get_settings
from common import metrics, load

settings = get_settings()
server_root = settings.vllm_host.rstrip("/")
if server_root.endswith("/v1"):
    server_root = server_root[:-3]

# Same load helper as Module 4: use the vllm CLI when present, otherwise fall
# back to the pure-Python load generator in common/load.py.
HAVE_VLLM_CLI = shutil.which("vllm") is not None

MANIFEST = "manifests/vllm-baseline.yaml"
print("namespace :", settings.namespace)
print("manifest  :", MANIFEST)
print("server    :", server_root)
print("load tool :", "vllm bench serve" if HAVE_VLLM_CLI else "pure-Python fallback (common/load.py)")


**What you should see:** your namespace, the manifest path, and the server root. The load tool line tells you which generator will run. Both report the same fields, so the lesson lands either way.

## 3. Read the current flags

Print the four flags from the manifest so you know your starting point. These are the numbers you will raise, and the values you will compare against after each change.

In [ ]:
# Show the current engine flags in the manifest.
with open(MANIFEST) as f:
    manifest_text = f.read()

for flag in ["--gpu-memory-utilization", "--max-model-len", "--max-num-seqs", "--max-num-batched-tokens"]:
    # The value is the line after the flag in the args list.
    lines = manifest_text.splitlines()
    for i, line in enumerate(lines):
        if flag in line and i + 1 < len(lines):
            print(f"{flag:<28} {lines[i+1].strip().strip(chr(34))}")
            break


**What you should see:** the four flags at their baseline values: `--gpu-memory-utilization` `0.40`, `--max-model-len` `2048`, `--max-num-seqs` `16`, `--max-num-batched-tokens` `2048`. The baseline is under-tuned on purpose so you have something real to fix.

## 4. The four flags and their tradeoffs

These are the four engine flags you will move. Each one trades GPU memory for concurrency in a different way:

- **`--gpu-memory-utilization`** (baseline `0.40`). The fraction of the card vLLM may use, most of which becomes KV cache. Raising it toward `0.90` buys more cache directly, which means more concurrent requests before preemption. Leave headroom; too close to `1.0` risks an out-of-memory under load.
- **`--max-num-seqs`** (baseline `16`). The cap on requests in the running batch. Raise it to let more requests run together, up to the point where the cache cannot hold them and preemption starts.
- **`--max-num-batched-tokens`** (baseline `2048`). The token budget per scheduler step. Raising it lets larger prefills batch together, which helps prompt-heavy traffic, at the cost of slightly longer steps.
- **`--max-model-len`** (baseline `2048`). The longest context the engine reserves cache for. Raise it only if your prompts or outputs need it, because each running request reserves blocks for the full length whether it uses them or not.

These are engine initialization arguments. vLLM reads them once at startup to size the cache and the scheduler, so changing any of them requires a fresh pod. That is why every pass through the loop includes a `rollout restart`.

## 5. The load sweep (same as Module 4)

This is the measurement helper from Module 4, restated so this module stands alone. It runs the load generator at one concurrency level and samples the KV cache and preemption counters while the run is in flight. `sweep()` warms the server up first, because right after a redeploy the pod is still loading the model and would skew the numbers. The KV gauge it reads is the one vLLM exposes as `kv_cache_usage_perc`; the code reads it under its key in the snapshot.

In [ ]:
# Requires a live vLLM endpoint. Uses the vllm CLI when present, otherwise the
# pure-Python load generator in common/load.py (same fields either way).
# Run one concurrency level and return throughput, latency, and pressure signals.
def run_level(concurrency, num_prompts=None, input_len=256, output_len=128):
    if not HAVE_VLLM_CLI:
        from common.config import build_client
        return load.run_level(
            build_client(settings), settings.model_name, settings.metrics_url,
            concurrency, num_prompts=num_prompts, output_len=output_len,
        )

    num_prompts = num_prompts or max(concurrency * 4, 16)
    peak = {"kv": 0.0, "waiting": 0.0}
    p0 = metrics.snapshot(settings.metrics_url)["vllm:num_preemptions_total"]
    stop = threading.Event()

    def sample():
        while not stop.is_set():
            try:
                s = metrics.snapshot(settings.metrics_url)
                peak["kv"] = max(peak["kv"], s["vllm:gpu_cache_usage_perc"])
                peak["waiting"] = max(peak["waiting"], s["vllm:num_requests_waiting"])
            except Exception:
                pass
            time.sleep(0.25)

    t = threading.Thread(target=sample, daemon=True); t.start()
    out_path = os.path.join(tempfile.gettempdir(), f"opt_c{concurrency}.json")
    cmd = [
        "vllm", "bench", "serve", "--backend", "openai-chat",
        "--base-url", server_root, "--endpoint", "/v1/chat/completions",
        "--model", settings.model_name, "--dataset-name", "random",
        "--random-input-len", str(input_len), "--random-output-len", str(output_len),
        "--num-prompts", str(num_prompts), "--max-concurrency", str(concurrency),
        "--save-result", "--result-filename", out_path,
    ]
    proc = subprocess.run(cmd, capture_output=True, text=True)
    stop.set(); t.join(timeout=2)
    if proc.returncode != 0:
        print(proc.stderr[-400:]); raise RuntimeError("bench failed")
    with open(out_path) as f:
        data = json.load(f)
    p1 = metrics.snapshot(settings.metrics_url)["vllm:num_preemptions_total"]
    return {
        "concurrency": concurrency,
        "output_throughput": data.get("output_throughput"),
        "mean_ttft_ms": data.get("mean_ttft_ms"),
        "mean_tpot_ms": data.get("mean_tpot_ms"),
        "peak_kv": peak["kv"], "peak_waiting": peak["waiting"],
        "preemptions": p1 - p0,
    }


def _warm_up(timeout_s=120):
    """After a redeploy the server needs a moment before it answers. Poll a
    tiny completion until it succeeds so the sweep measures a ready server."""
    from common.config import build_client
    client = build_client(settings).with_options(timeout=10, max_retries=0)
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        try:
            client.chat.completions.create(
                model=settings.model_name,
                messages=[{"role": "user", "content": "ok"}], max_tokens=1,
            )
            return True
        except Exception:
            time.sleep(3)
    return False


def _num(value, width):
    # Right-aligned number, or "n/a" when a level returned no data.
    return format(value, f">{width}.0f") if value is not None else format("n/a", f">{width}")


def sweep(levels=(1, 16, 64, 128), label=""):
    _warm_up()
    rows = []
    for c in levels:
        r = run_level(c); rows.append(r)
        kv = (r["peak_kv"] or 0) * 100
        print(f"[{label}] c={c:>3}  {_num(r['output_throughput'], 7)} tok/s  "
              f"TTFT={_num(r['mean_ttft_ms'], 6)}ms  KV={kv:>4.0f}%  "
              f"preempt={_num(r['preemptions'], 4)}")
    return rows


**What you should see:** nothing yet. This defines `run_level` and `sweep`. Each sweep takes a few minutes because it drives real load at four concurrency levels.

## 6. Measure the baseline (before)

Run the sweep against the under-tuned server and save it as your before picture. This is the fail-first beat: with `--gpu-memory-utilization` at `0.40`, most of the card sits idle while a tiny cache forces requests to queue. You should see the same early knee and KV pressure you found in Module 4, printed on screen so the problem is visible before you fix it.

In [ ]:
# Requires a live vLLM endpoint.
# Baseline sweep against the under-tuned server.
before = sweep(label="before")


**What you should see:** modest peak throughput, KV cache climbing toward 100 percent at the top of the sweep, and preemptions greater than zero. The card is mostly empty, but the cache is the bottleneck. That is the gap you close next.

## 7. Change the biggest lever first

Take the largest lever first. Open `manifests/vllm-baseline.yaml`, find `--gpu-memory-utilization`, and change `0.40` to `0.90`. You can edit the file in the JupyterLab editor, or run the cell below to patch it in place. `set_flag` rewrites the value on the line after the flag while keeping the YAML list marker `- ` and the indent, because dropping either turns the args block into invalid YAML that `kubectl apply` will reject.

In [ ]:
# Patch one flag value in the manifest file in place.
def set_flag(flag, new_value):
    with open(MANIFEST) as f:
        lines = f.readlines()
    for i, line in enumerate(lines):
        if flag in line and i + 1 < len(lines):
            # The value sits on the next line as a YAML list item: '- "0.40"'.
            # Preserve the indent AND the '- ' marker, or the manifest becomes
            # invalid YAML and kubectl apply rejects it.
            indent = lines[i+1][:len(lines[i+1]) - len(lines[i+1].lstrip())]
            lines[i+1] = f'{indent}- "{new_value}"\n'
            break
    with open(MANIFEST, "w") as f:
        f.writelines(lines)
    print(f"set {flag} = {new_value}")

set_flag("--gpu-memory-utilization", "0.90")


**What you should see:** a confirmation that the flag is now `0.90`. The change is on disk but not yet live. The running pod is still serving the old value until you redeploy.

## 8. Redeploy and wait

Apply the manifest, then restart the deployment so a fresh pod reads the new engine arguments. `kubectl apply` updates the spec, but because these are startup arguments, you need a new pod, and `rollout restart` gives you one. Wait for it to report ready before measuring, or you will benchmark a pod that is still loading the model.

The manifest sets `strategy: Recreate` instead of the default rolling update. A single-GPU pod holds the whole card, so a rolling update would deadlock: the new pod cannot get the GPU until the old one frees it, and a rolling update will not free the old one until the new one is ready. `Recreate` terminates the old pod first, then starts the new one. The manifest also sets `enableServiceLinks: false`, because the Service is named `vllm` and Kubernetes would otherwise inject `VLLM_PORT=tcp://<ip>:8000`, which vLLM misreads as its own port and refuses to start.

In [ ]:
# Requires a live cluster.
# Apply the new manifest, restart the pod, and wait until it is ready.
ns = settings.namespace
subprocess.run(["kubectl", "apply", "-f", MANIFEST, "-n", ns], check=True)
subprocess.run(["kubectl", "rollout", "restart", "deploy/vllm", "-n", ns], check=True)
subprocess.run(["kubectl", "rollout", "status", "deploy/vllm", "-n", ns, "--timeout=300s"], check=True)
print("rollout complete; pod is serving the new config")


**What you should see:** `kubectl` apply and restart messages, then a rollout status that ends with the deployment successfully rolled out. Loading the model can take a minute or two, which is why the timeout is generous.

> NOTE: If the rollout never becomes ready, you may have set `--gpu-memory-utilization` too high. Check `kubectl logs deploy/vllm -n $NAMESPACE` for an out-of-memory error and back it off by `0.05`.

## 9. Measure again (after raising memory)

Run the same sweep against the redeployed server and compare. With more of the GPU available for the KV cache, you should see higher peak throughput, the knee pushed to a higher concurrency, and preemptions starting later or not at all.

In [ ]:
# Requires a live vLLM endpoint.
# Sweep again after raising gpu-memory-utilization.
after_mem = sweep(label="after-mem")


**What you should see:** higher throughput at the upper concurrency levels and a lower peak KV percentage for the same load, because the cache is now much larger. On a card that is large relative to a 4B model, this flag alone may move the numbers only a little, because the batch cap, not the cache, becomes the next limit. The proof of where the ceiling really sits comes in the next step.

## 10. Raise the batch size cap

Now that the cache is bigger, let more requests run at once. Raise `--max-num-seqs` from `16` to `64` and `--max-num-batched-tokens` from `2048` to `8192`, redeploy, and sweep again. Watch for the point where pushing the batch higher starts causing preemption again. That is the new limit, and on a card with memory to spare this is the change that actually moves throughput.

In [ ]:
# Requires a live cluster and endpoint.
# Raise the batch caps, redeploy, and sweep once more.
set_flag("--max-num-seqs", "64")
set_flag("--max-num-batched-tokens", "8192")

subprocess.run(["kubectl", "apply", "-f", MANIFEST, "-n", ns], check=True)
subprocess.run(["kubectl", "rollout", "restart", "deploy/vllm", "-n", ns], check=True)
subprocess.run(["kubectl", "rollout", "status", "deploy/vllm", "-n", ns, "--timeout=300s"], check=True)

after_batch = sweep(label="after-batch")


**What you should see:** throughput at high concurrency higher again, because the batch can hold more requests now that the cache can back them. On the workshop hardware, a 4B instruct model on a single RTX 4000 Ada with 20GB, raising `--max-num-seqs` from 16 to 64 lifted throughput by roughly 3.4x, while the memory flag alone had barely moved it: on a big card relative to a small model, the batch cap was the limit, not KV memory. If preemptions reappear at the top, you have found the ceiling for this model and this GPU.

## 11. Compare before and after

Put the runs side by side. The story you want is the same or better latency at low concurrency, and clearly higher throughput at high concurrency, with the knee pushed right.

In [ ]:
# Plot throughput vs concurrency for each configuration.
import matplotlib.pyplot as plt

def xs(rows): return [r["concurrency"] for r in rows]
def thru(rows): return [r["output_throughput"] for r in rows]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(xs(before), thru(before), marker="o", label="before (gpu-mem 0.40)")
ax.plot(xs(after_mem), thru(after_mem), marker="o", label="after gpu-mem 0.90")
ax.plot(xs(after_batch), thru(after_batch), marker="o", label="after batch caps raised")
ax.set_xlabel("concurrency"); ax.set_ylabel("output tokens/s")
ax.set_xscale("log", base=2); ax.set_title("Throughput before and after tuning")
ax.legend(); ax.grid(True, alpha=0.3); fig.tight_layout()

peak_before = max(r["output_throughput"] for r in before)
peak_after = max(r["output_throughput"] for r in after_batch)
print(f"peak throughput: {peak_before:.0f} -> {peak_after:.0f} tok/s "
      f"({peak_after/peak_before:.1f}x)")


**What you should see:** the after curves sitting above the before curve at high concurrency, and a peak-throughput multiple printed at the bottom. On the workshop hardware this came out around 3.4x, driven almost entirely by the batch caps. You got that without touching the model or the hardware.

## Things to know

- **The metrics pick the next flag, not you.** Preemptions greater than zero with KV near 100 percent means you are cache bound: raise `--gpu-memory-utilization`, then reach for quantization (Module 6). KV with headroom but flat throughput means the batch is too small: raise `--max-num-seqs` and `--max-num-batched-tokens`. TTFT climbing while throughput holds means prefills are queueing: raise `--max-num-batched-tokens`.
- **The KV gauge was renamed.** vLLM's V1 engine renamed the cache gauge from `gpu_cache_usage_perc` to `kv_cache_usage_perc`. `snapshot()` in `common/metrics.py` returns it under both keys, so the sweep code reads either one without breaking.
- **`Recreate`, not rolling update.** One pod holds the whole GPU, so a rolling update deadlocks waiting for a card that never frees. The manifest sets `strategy: Recreate` to terminate the old pod before starting the new one.
- **`enableServiceLinks: false` is load-bearing.** A Service named `vllm` makes Kubernetes inject `VLLM_PORT`, which vLLM misreads as its own port and refuses to start. The manifest disables service-link env vars to avoid it.
- **One change, one redeploy, one sweep.** Change four flags at once and the metrics cannot tell you which one helped. Discipline is what makes this loop work.

## Try it yourself

**Push `--max-num-seqs` until it breaks.** Raise it past `64` to `128`, redeploy, and sweep. Find the value where preemptions reappear at the top of the sweep. That value is the real ceiling for this model and GPU. **Stretch:** back it off by 16 and confirm preemptions go back to zero.

**Trade context for concurrency.** Lower `--max-model-len` to `1024` and watch peak KV drop at the same load, then raise `--max-num-seqs` into the room you just freed. Each running request reserves cache for the full context length, so shorter context buys you batch.

**Read before you change.** Run a sweep, decide the next flag from the metrics alone before you look at the suggestions in Things to know, then check whether you agreed.

In [ ]:
# Change one flag, then run the cell. Redeploy with the cell in section 8 to apply it.
set_flag("--max-num-seqs", "128")     # push the batch cap; watch for preemptions
# set_flag("--max-model-len", "1024")  # or trade context length for batch room

# After changing a flag, re-apply and restart to make it live:
# subprocess.run(["kubectl", "apply", "-f", MANIFEST, "-n", ns], check=True)
# subprocess.run(["kubectl", "rollout", "restart", "deploy/vllm", "-n", ns], check=True)
# subprocess.run(["kubectl", "rollout", "status", "deploy/vllm", "-n", ns, "--timeout=300s"], check=True)
# after = sweep(label="after-128")


## Reset (optional)

If you want to hand the environment back at the baseline, set the flags to their original values and redeploy. Otherwise leave it tuned; the later modules are happy with a well-configured server.

In [ ]:
# Optional: restore the baseline flags and redeploy.
# Uncomment to run.
# for flag, val in [("--gpu-memory-utilization", "0.40"), ("--max-model-len", "2048"),
#                   ("--max-num-seqs", "16"), ("--max-num-batched-tokens", "2048")]:
#     set_flag(flag, val)
# subprocess.run(["kubectl", "apply", "-f", MANIFEST, "-n", ns], check=True)
# subprocess.run(["kubectl", "rollout", "restart", "deploy/vllm", "-n", ns], check=True)
print("leave tuned, or uncomment above to reset")


## Summary

- You ran the measure, read, change, redeploy, re-measure loop on a server you control, and proved the gain against the same sweep.
- The metrics, not a hunch, picked each flag: cache pressure called for memory, flat throughput called for a bigger batch.
- Raising the batch caps moved throughput most on a card large relative to the model, roughly 3.4x on the workshop hardware, while the memory flag alone barely moved it.
- Redeploying needs `Recreate` and `enableServiceLinks: false`, because one pod owns the whole GPU and a Service named `vllm` would otherwise break startup.
- The bottleneck from Module 4 moved because you moved it, with no model change and no new hardware.

## Next

**Module 6: Quantization with LLM Compressor.** Config tuning eventually runs out of room: the batch caps hit the ceiling this GPU and model can hold. Next you shrink the model itself, quantize it with LLM Compressor, serve the smaller model, and measure both the extra KV headroom you gain and the accuracy you give up.